# CareLoop Clinics — No-Show Guard
## Part 3: Machine Learning (Predict No-Shows and Choose the Model Honestly)

### Task 3.1 — Features, split and baseline

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score
)

df = pd.read_csv("appointments_clean.csv")
appt_ids = df["appointment_id"].copy()

feature_cols = [
    "age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
    "reminder_sent", "booking_channel", "visit_type"
]
X = df[feature_cols].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

baseline_acc = (y_test == 0).mean()
print("X shape:", X.shape)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"No-shows in test set: {y_test.sum()}")
print(f"Baseline accuracy (always predict Attended): {baseline_acc:.4f}")

### Task 3.2 — Logistic regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_preds = lr.predict(X_test_s)
lr_probs = lr.predict_proba(X_test_s)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, lr_preds):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, lr_preds))
print(f"Precision: {precision_score(y_test, lr_preds):.4f}")
print(f"Recall:    {recall_score(y_test, lr_preds):.4f}")
print(f"F1:        {f1_score(y_test, lr_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, lr_probs):.4f}")

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": lr.coef_[0],
    "scale": scaler.scale_
})
print("\nCoefficients:\n", coef_df)

rem_row = coef_df[coef_df["feature"] == "reminder_sent"].iloc[0]
odds_scaled = np.exp(rem_row["coef"])
odds_unscaled = np.exp(rem_row["coef"] / rem_row["scale"])
print(f"\nexp(coef) for 1-SD change in reminder_sent: {odds_scaled:.4f}")
print(f"exp(coef / scale) for No (0) -> Yes (1):     {odds_unscaled:.4f}")

In this model, holding the other features constant, appointments with a reminder sent are associated with roughly 69% lower odds of a no-show (odds ratio 0.31); the two numbers differ because `exp(coef)` reflects a 1-standard-deviation change in the scaled feature (~0.495) whereas `exp(coef / scale)` converts back to the full 0-to-1 (`No` to `Yes`) unit change. Because the data are observational, this is an association and does not prove that reminders cause fewer no-shows.

### Task 3.3 — Decision tree and overfitting

In [ ]:
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
print(f"Full tree (depth={dt_full.get_depth()}): train acc={dt_full.score(X_train, y_train):.4f}, test acc={dt_full.score(X_test, y_test):.4f}")

dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)
dt3_preds = dt_3.predict(X_test)
dt3_probs = dt_3.predict_proba(X_test)[:, 1]
print(f"Depth-3 tree: train acc={dt_3.score(X_train, y_train):.4f}, test acc={dt_3.score(X_test, y_test):.4f}")

The full-depth tree achieves 1.00 training accuracy but drops to ~0.67 on the test set (below the 0.7525 baseline), showing severe overfitting where it memorised training noise, whereas limiting `max_depth=3` narrows the gap and generalises better.

### Task 3.4 — Random forest and a comparison table

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]

print(f"Random Forest: train acc={rf.score(X_train, y_train):.4f}, test acc={accuracy_score(y_test, rf_preds):.4f}, "
      f"recall={recall_score(y_test, rf_preds):.4f}, ROC-AUC={roc_auc_score(y_test, rf_probs):.4f}")

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 3 feature importances:\n", imp.head(3))

comparison = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "accuracy": round(accuracy_score(y_test, lr_preds), 4),
        "recall": round(recall_score(y_test, lr_preds), 4),
        "roc_auc": round(roc_auc_score(y_test, lr_probs), 4),
    },
    {
        "model": "Decision Tree (max_depth=3)",
        "accuracy": round(accuracy_score(y_test, dt3_preds), 4),
        "recall": round(recall_score(y_test, dt3_preds), 4),
        "roc_auc": round(roc_auc_score(y_test, dt3_probs), 4),
    },
    {
        "model": "Random Forest (100 trees)",
        "accuracy": round(accuracy_score(y_test, rf_preds), 4),
        "recall": round(recall_score(y_test, rf_preds), 4),
        "roc_auc": round(roc_auc_score(y_test, rf_probs), 4),
    },
])
print("\nTest-set comparison (for reporting only):\n", comparison)

Logistic regression has the highest test ROC-AUC and accuracy of the three models (the formal model choice is made strictly from training cross-validation in Task 3.5).

### Task 3.5 — Tune, choose the model and the threshold, then test once

In [ ]:
# (a) GridSearchCV on training set
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5,
    scoring="roc_auc"
)
grid.fit(X_train, y_train)
print("Best RF params:", grid.best_params_)
print(f"Best RF CV ROC-AUC: {grid.best_score_:.4f}")

# (b) 5-fold CV ROC-AUC on training set for all candidates
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])
lr_cv_auc = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
dt3_cv_auc = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
rf_cv_auc = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()

print(f"CV ROC-AUC -> LR: {lr_cv_auc:.4f} | DT(3): {dt3_cv_auc:.4f} | RF(100): {rf_cv_auc:.4f} | Tuned RF: {grid.best_score_:.4f}")

# (c) Out-of-fold probabilities for winner (Logistic Regression pipeline)
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
for thresh in [0.5, 0.35]:
    oof_flag = (oof >= thresh).astype(int)
    print(
        f"OOF threshold {thresh}: precision={precision_score(y_train, oof_flag):.4f}, "
        f"recall={recall_score(y_train, oof_flag):.4f}, f1={f1_score(y_train, oof_flag):.4f}, "
        f"flagged={oof_flag.sum()}"
    )

chosen_threshold = 0.35

# (d) Fit winner on full training set and evaluate ONCE on test set
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_flag = (test_probs >= chosen_threshold).astype(int)

print("\nFinal Test Evaluation (Logistic Regression @ 0.35):")
print(f"Precision: {precision_score(y_test, test_flag):.4f}")
print(f"Recall:    {recall_score(y_test, test_flag):.4f}")
print(f"Flagged:   {test_flag.sum()} / {len(test_flag)}")
print(f"ROC-AUC:   {roc_auc_score(y_test, test_probs):.4f}")

# (e) Save test_risk_scores.csv
risk_df = pd.DataFrame({
    "appointment_id": appt_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": np.round(test_probs, 6),
    "flag": test_flag
})
risk_df.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv with", len(risk_df), "rows and", risk_df["flag"].sum(), "flagged.")

In practice, a clinic chooses the alert threshold based on how many evening follow-up calls the front-desk staff can make, weighed against the cost of an empty doctor slot.